In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

In [6]:
url = "https://volcano.si.edu/reports_weekly.cfm?vtab=feeds"

response = requests.get(url)
response

<Response [200]>

In [7]:
soup = BeautifulSoup(response.content)
table = soup.find('table')
table

<table>
<thead>
<tr>
<th colspan="5">Smithsonian / USGS Weekly Volcanic Activity Report for the week of 19 March-25 March 2025</th>
</tr>
<tr>
<th>Name</th>
<th>Country</th>
<th>Volcanic Region</th>
<th>Eruption Start Date</th>
<th class="aligncenter">Report Status</th>
</tr>
</thead>
<tbody>
<tr>
<td class="textbold"><a href="#vn_264180">Lewotobi</a></td>
<td class="textbold">Indonesia</td>
<td class="textbold">Sunda Volcanic Arc</td>
<td class="textbold">2023 Dec 23</td>
<td class="textbold aligncenter"><a data-tooltip="tt264180" style="color:red;">New</a></td>
</tr>
<tr>
<td class="textbold"><a href="#vn_264230">Lewotolok</a></td>
<td class="textbold">Indonesia</td>
<td class="textbold">Sunda Volcanic Arc</td>
<td class="textbold">2025 Jan 16</td>
<td class="textbold aligncenter"><a data-tooltip="tt264230" style="color:red;">New</a></td>
</tr>
<tr>
<td class="textbold"><a href="#vn_345040">Poas</a></td>
<td class="textbold">Costa Rica</td>
<td class="textbold">Central America Volcan

In [8]:
# Extract column names
headers = [th.get_text(strip=True) for th in table.find_all('th')][1:]
headers

['Name', 'Country', 'Volcanic Region', 'Eruption Start Date', 'Report Status']

In [9]:
volcano_data = []
headers = [th.get_text(strip=True) for th in table.find_all('th')][1:]

for row in table.find_all('tr')[2:]:  # Skip header row
    cols = row.find_all(['td', 'th'])

    print(f"Row columns count: {len(cols)}")
    print(f"Row columns text: {[col.get_text(strip=True) for col in cols]}")
    
    if len(cols) < len(headers):
        print("Skipping row: not enough columns")
        continue

    try:
        volcano_link = row.find('a', href=re.compile(r'#vn_'))
        
        if not volcano_link:
            print("No volcano link found in this row")
            continue
        
        volcano_id = volcano_link['href'].split('#vn_')[1]
        volcano_name = volcano_link.get_text(strip=True)
        start_date = cols[3].get_text(strip=True)
        
        report_status = row.find("a", attrs={"data-tooltip": True})
        report_text = report_status.get_text(strip=True) if report_status else None

        row_data = {
              'volcano_id': volcano_id,
              'volcano_name': volcano_name,
              'start_date': start_date,
              'report_status': report_text
        }
        
        # # Add other column data
        # for i, col in enumerate(cols):
        #     if i < len(headers):
        #         row_data[headers[i]] = col.get_text(strip=True)
        
        volcano_data.append(row_data)
        print(f"Added data for: {volcano_name}")
    
    except Exception as e:
        print(f"Error processing row: {e}")

df = pd.DataFrame(volcano_data)
df

Row columns count: 5
Row columns text: ['Lewotobi', 'Indonesia', 'Sunda Volcanic Arc', '2023 Dec 23', 'New']
Added data for: Lewotobi
Row columns count: 5
Row columns text: ['Lewotolok', 'Indonesia', 'Sunda Volcanic Arc', '2025 Jan 16', 'New']
Added data for: Lewotolok
Row columns count: 5
Row columns text: ['Poas', 'Costa Rica', 'Central America Volcanic Arc', '2025 Jan 5', 'New']
Added data for: Poas
Row columns count: 5
Row columns text: ['Ahyi', 'United States', 'Mariana Volcanic Arc', '2024 Aug 5', 'Continuing']
Added data for: Ahyi
Row columns count: 5
Row columns text: ['Aira', 'Japan', 'Ryukyu Volcanic Arc', '2017 Mar 25', 'Continuing']
Added data for: Aira
Row columns count: 5
Row columns text: ['Bezymianny', 'Russia', 'Eastern Kamchatka Volcanic Arc', '2024 Dec 24', 'Continuing']
Added data for: Bezymianny
Row columns count: 5
Row columns text: ['Dukono', 'Indonesia', 'Halmahera Volcanic Arc', '1933 Aug 13', 'Continuing']
Added data for: Dukono
Row columns count: 5
Row column

,volcano_id,volcano_name,start_date,report_status
0,264180,Lewotobi,2023 Dec 23,New
1,264230,Lewotolok,2025 Jan 16,New
2,345040,Poas,2025 Jan 5,New
3,284141,Ahyi,2024 Aug 5,Continuing
4,282080,Aira,2017 Mar 25,Continuing
5,300250,Bezymianny,2024 Dec 24,Continuing
6,268010,Dukono,1933 Aug 13,Continuing
7,211060,Etna,2022 Nov 27,Continuing
8,311120,Great Sitkin,2021 May 25,Continuing
9,243080,Home Reef,2024 Dec 4,Continuing


In [10]:
def scrap_volcanic_weekly_report():

    url = "https://volcano.si.edu/reports_weekly.cfm?vtab=feeds"
    response = requests.get(url)

    soup = BeautifulSoup(response.content)
    table = soup.find('table')

    volcano_data = []
    headers = [th.get_text(strip=True) for th in table.find_all('th')][1:]

    for row in table.find_all('tr')[2:]:  # Skip header row
        cols = row.find_all(['td', 'th'])

        if len(cols) < len(headers):
            print("Skipping row: not enough columns")
            continue

        try:
            volcano_link = row.find('a', href=re.compile(r'#vn_'))
            
            if not volcano_link:
                print("No volcano link found in this row")
                continue
            
            volcano_id = volcano_link['href'].split('#vn_')[1]
            volcano_name = volcano_link.get_text(strip=True)
            start_date = cols[3].get_text(strip=True)
            
            report_status = row.find("a", attrs={"data-tooltip": True})
            report_text = report_status.get_text(strip=True) if report_status else None

            row_data = {
                'volcano_id': volcano_id,
                'volcano_name': volcano_name,
                'start_date': start_date,
                'report_status': report_text
            }
            
            volcano_data.append(row_data)
        
        except Exception as e:
            print(f"Error processing row: {e}")

    volcanic_weekly_report = pd.DataFrame(volcano_data)

    return volcanic_weekly_report
    
volcanic_weekly_report = scrap_volcanic_weekly_report()
volcanic_weekly_report

,volcano_id,volcano_name,start_date,report_status
0,264180,Lewotobi,2023 Dec 23,New
1,264230,Lewotolok,2025 Jan 16,New
2,345040,Poas,2025 Jan 5,New
3,284141,Ahyi,2024 Aug 5,Continuing
4,282080,Aira,2017 Mar 25,Continuing
5,300250,Bezymianny,2024 Dec 24,Continuing
6,268010,Dukono,1933 Aug 13,Continuing
7,211060,Etna,2022 Nov 27,Continuing
8,311120,Great Sitkin,2021 May 25,Continuing
9,243080,Home Reef,2024 Dec 4,Continuing


In [ ]:
volcano_link = row.find('a', href=re.compile(r'#vn_'))
volcano_link

<a href="#vn_241040">Whakaari/White Island</a>

In [ ]:
row.find('a', href=re.compile(r'#vn_'))

<a href="#vn_241040">Whakaari/White Island</a>

In [ ]:
volcano_number = volcano_link['href'].split('#vn_')[1]
volcano_number

'241040'

In [ ]:
volcano_name = volcano_link.get_text(strip=True)
volcano_name

'Whakaari/White Island'

In [ ]:
start_date = cols[3].get_text(strip=True)
start_date

'2024 May 24'

In [ ]:
report_status = row.find("a", attrs={"data-tooltip": True})
report_status

<a data-tooltip="tt241040" style="color:green;">Continuing</a>

In [ ]:
report_text = report_status.get_text(strip=True) if report_status else None
report_text

'Continuing'

In [ ]:
volcano_data

[{'volcano_id': '264180',
  'volcano_name': 'Lewotobi',
  'start_date': '2023 Dec 23',
  'report_status': 'New'},
 {'volcano_id': '264230',
  'volcano_name': 'Lewotolok',
  'start_date': '2025 Jan 16',
  'report_status': 'New'},
 {'volcano_id': '345040',
  'volcano_name': 'Poas',
  'start_date': '2025 Jan 5',
  'report_status': 'New'},
 {'volcano_id': '284141',
  'volcano_name': 'Ahyi',
  'start_date': '2024 Aug 5',
  'report_status': 'Continuing'},
 {'volcano_id': '282080',
  'volcano_name': 'Aira',
  'start_date': '2017 Mar 25',
  'report_status': 'Continuing'},
 {'volcano_id': '300250',
  'volcano_name': 'Bezymianny',
  'start_date': '2024 Dec 24',
  'report_status': 'Continuing'},
 {'volcano_id': '268010',
  'volcano_name': 'Dukono',
  'start_date': '1933 Aug 13',
  'report_status': 'Continuing'},
 {'volcano_id': '211060',
  'volcano_name': 'Etna',
  'start_date': '2022 Nov 27',
  'report_status': 'Continuing'},
 {'volcano_id': '311120',
  'volcano_name': 'Great Sitkin',
  'start_d

#### Scraping Yearly Report 2025

In [11]:
url = "https://volcano.si.edu/faq/index.cfm?question=eruptionsbyyear&checkyear=2025"
response = requests.get(url)

In [12]:
soup = BeautifulSoup(response.content)
table = soup.find('table')
table

<table>
<thead>
<tr>
<th>Volcano</th>
<th>Country</th>
<th>Eruption Start Date</th>
<th>Eruption Stop Date</th>
<th>Max VEI</th>
</tr>
</thead>
<tbody>
<tr>
<td class="textbold"><a href="/volcano.cfm?vn=351060">Purace</a></td>
<td>Colombia</td>
<td>2025 Jan 19</td>
<td>2025 Feb 21</td>
<td>–</td>
</tr>
<tr>
<td class="textbold"><a href="/volcano.cfm?vn=264230">Lewotolok</a></td>
<td>Indonesia</td>
<td>2025 Jan 16</td>
<td>2025 Feb 21 (continuing)</td>
<td>–</td>
</tr>
<tr>
<td class="textbold"><a href="/volcano.cfm?vn=344040">Telica</a></td>
<td>Nicaragua</td>
<td>2025 Jan 11</td>
<td>2025 Feb 21 (continuing)</td>
<td>–</td>
</tr>
<tr>
<td class="textbold"><a href="/volcano.cfm?vn=345040">Poas</a></td>
<td>Costa Rica</td>
<td>2025 Jan 5</td>
<td>2025 Feb 21 (continuing)</td>
<td>–</td>
</tr>
<tr>
<td class="textbold"><a href="/volcano.cfm?vn=300250">Bezymianny</a></td>
<td>Russia</td>
<td>2024 Dec 24</td>
<td>2025 Feb 21 (continuing)</td>
<td>–</td>
</tr>
<tr>
<td class="textbold"><a h

In [13]:
# Extract column names
headers = [th.get_text(strip=True) for th in table.find_all('th')]
headers

['Volcano', 'Country', 'Eruption Start Date', 'Eruption Stop Date', 'Max VEI']

In [69]:
soup = BeautifulSoup(response.text, 'html.parser')

# Find the table - you might need to adjust the selector based on the actual page structure
table = soup.find('table')

# Extract table data
data = []

# Get table headers
headers = []
for th in table.find_all('th'):
    headers.append(th.text.strip())

# Get table rows
for row in table.find_all('tr')[1:]:  # Skip the header row
    row_data = []
    for td in row.find_all('td'):
        row_data.append(td.text.strip())
    
    if row_data:  # Skip empty rows
        data.append(row_data)

# Create a DataFrame
df = pd.DataFrame(data, columns=headers)
df

,Volcano,Country,Eruption Start Date,Eruption Stop Date,Max VEI
0,Purace,Colombia,2025 Jan 19,2025 Feb 21,–
1,Lewotolok,Indonesia,2025 Jan 16,2025 Feb 21 (continuing),–
2,Telica,Nicaragua,2025 Jan 11,2025 Feb 21 (continuing),–
3,Poas,Costa Rica,2025 Jan 5,2025 Feb 21 (continuing),–
4,Bezymianny,Russia,2024 Dec 24,2025 Feb 21 (continuing),–
5,Kilauea,United States,2024 Dec 23,2025 Feb 21 (continuing),–
6,Dieng Volcanic Complex,Indonesia,2024 Dec 18,2025 Jan 6,–
7,Home Reef,Tonga,2024 Dec 4,2025 Feb 21 (continuing),–
8,Dempo,Indonesia,2024 Nov 23,2025 Feb 21 (continuing),–
9,Kanlaon,Philippines,2024 Oct 19,2025 Feb 21 (continuing),–


In [82]:
def scrap_yearly_report():
    url = "https://volcano.si.edu/faq/index.cfm?question=eruptionsbyyear&checkyear=2025"
    response = requests.get(url)

    soup = BeautifulSoup(response.text, 'html.parser')

    # Find the table - you might need to adjust the selector based on the actual page structure
    table = soup.find('table')

    # Extract table data
    data = []

    # Get table headers
    headers = []
    for th in table.find_all('th'):
        headers.append(th.text.strip())

    # Get table rows
    for row in table.find_all('tr')[1:]:  # Skip the header row
        row_data = []
        for td in row.find_all('td'):
            row_data.append(td.text.strip())
        
        if row_data:  # Skip empty rows
            data.append(row_data)

    # Create a DataFrame
    yearly_report = pd.DataFrame(data, columns=headers)
    yearly_report.columns = [column.lower() for column in yearly_report.columns]
    return yearly_report  

In [94]:
def clean_yearly_report():
    date_column = 'eruption stop date'
    
    df = scrap_yearly_report()
    # Create the status column with default 'Over'
    df['status'] = 'Over'
    
    # Update status based on the continuing text
    mask = df[date_column].str.contains('\(continuing\)', regex=True, na=False)
    df.loc[mask, 'status'] = 'On going'
    
    # Clean up the date strings
    df[date_column] = df[date_column].str.replace(r'\s*\(continuing\)\s*', '', regex=True)
    
    # Count and print how many ongoing eruptions were found
    ongoing_count = df['status'].value_counts().get('On going', 0)
    print(f"Found {ongoing_count} ongoing eruptions")

    df = df[['volcano', 'country', 'eruption start date', 'eruption stop date', 'status', 'max vei']]
    df.rename(columns={'volcano':'volcano_name'}, inplace=True)
    
    return df


<>:9: SyntaxWarning: invalid escape sequence '\('
<>:9: SyntaxWarning: invalid escape sequence '\('
/var/folders/yz/shh48k7s5kg3qtp0x9hzjb6c0000gn/T/ipykernel_77563/3504332965.py:9: SyntaxWarning: invalid escape sequence '\('
  mask = df[date_column].str.contains('\(continuing\)', regex=True, na=False)


In [95]:
yearly_report = scrap_yearly_report()
yearly_report = clean_yearly_report()
yearly_report

Found 44 ongoing eruptions


,volcano_name,country,eruption start date,eruption stop date,status,max vei
0,Purace,Colombia,2025 Jan 19,2025 Feb 21,Over,–
1,Lewotolok,Indonesia,2025 Jan 16,2025 Feb 21,On going,–
2,Telica,Nicaragua,2025 Jan 11,2025 Feb 21,On going,–
3,Poas,Costa Rica,2025 Jan 5,2025 Feb 21,On going,–
4,Bezymianny,Russia,2024 Dec 24,2025 Feb 21,On going,–
5,Kilauea,United States,2024 Dec 23,2025 Feb 21,On going,–
6,Dieng Volcanic Complex,Indonesia,2024 Dec 18,2025 Jan 6,Over,–
7,Home Reef,Tonga,2024 Dec 4,2025 Feb 21,On going,–
8,Dempo,Indonesia,2024 Nov 23,2025 Feb 21,On going,–
9,Kanlaon,Philippines,2024 Oct 19,2025 Feb 21,On going,–
